## Imports & Configurations

In [1]:
from astropy.table import Table
import numpy as np
import pandas as pd
import os
import warnings

## Data Paths

In [2]:
warnings.filterwarnings('ignore')
data_path = os.path.join(os.getcwd(), '..', 'Data')

## Load DR16Q v4 and Correct for Galactic Extinction

In [3]:
qso_filename = os.path.join(data_path, 'DR16Q_v4.fits')
qso_data = Table.read(qso_filename, format='fits', hdu=1)

u_qso=qso_data['PSFMAG'][:,0]
g_qso=qso_data['PSFMAG'][:,1]
r_qso=qso_data['PSFMAG'][:,2]
i_qso=qso_data['PSFMAG'][:,3]
z_qso=qso_data['PSFMAG'][:,4]

u_qso_err = qso_data['PSFMAGERR'][:, 0]
g_qso_err = qso_data['PSFMAGERR'][:, 1]
r_qso_err = qso_data['PSFMAGERR'][:, 2]
i_qso_err = qso_data['PSFMAGERR'][:, 3]
z_qso_err = qso_data['PSFMAGERR'][:, 4]

u_qso_extinction = qso_data['EXTINCTION'][:, 0]
g_qso_extinction = qso_data['EXTINCTION'][:, 1]
r_qso_extinction = qso_data['EXTINCTION'][:, 2]
i_qso_extinction = qso_data['EXTINCTION'][:, 3]
z_qso_extinction = qso_data['EXTINCTION'][:, 4]

u_dered_qso = u_qso - u_qso_extinction
g_dered_qso = g_qso - g_qso_extinction
r_dered_qso = r_qso - r_qso_extinction
i_dered_qso = i_qso - i_qso_extinction
z_dered_qso = z_qso - z_qso_extinction

## Quasar Sample Selection

In [4]:
n_total_qso = len(qso_data)

confirmed_qso = ((qso_data['IS_QSO_FINAL'] == 1) & (qso_data['THING_ID'] != -1) & (qso_data['Z'] != -999) & (qso_data['Z_PCA'] >= 0))
n_after_classification = np.count_nonzero(confirmed_qso)

no_z_warning = confirmed_qso & (qso_data['ZWARNING'] == 0) & (qso_data['ZWARN_PCA'] == 0)
n_after_zwarning = np.count_nonzero(no_z_warning)

pipe_only_high_z = (qso_data['SOURCE_Z'] == 'PIPE') & (qso_data['Z'] > 5)
not_pipe_only_high_z = no_z_warning & (~pipe_only_high_z)
n_after_pipe_only_high_z = np.count_nonzero(not_pipe_only_high_z)

# Exclude quasars whose pipeline (Z_PIPE) and QuasarNET (Z_QN) redshifts differ by
# more than 10,000 km/s, using dv = c |Z_QN - Z_PIPE| / (1 + Z_PIPE); applied only
# where both redshifts are valid (>= 0).
# Lyke et al. (2020, Section 4.4) list these among the objects without reliable PCA redshifts.
c_kms = 299792.458
z_qn = np.asarray(qso_data['Z_QN'], dtype=float)
z_pipe = np.asarray(qso_data['Z_PIPE'], dtype=float)
with np.errstate(divide='ignore', invalid='ignore'):
    dv_qn_pipe = c_kms * np.abs(z_qn - z_pipe) / (1 + z_pipe)
inconsistent_z = (z_qn >= 0) & (z_pipe >= 0) & (dv_qn_pipe > 10000)
consistent_z = not_pipe_only_high_z & (~inconsistent_z)
n_after_consistent = np.count_nonzero(consistent_z)

mask_phot_err = ((u_qso_err >= 0) & (u_qso_err < 0.1) & (g_qso_err >= 0) & (g_qso_err < 0.1) & (r_qso_err >= 0) & (r_qso_err < 0.1) &
                 (i_qso_err >= 0) & (i_qso_err < 0.1) & (z_qso_err >= 0) & (z_qso_err < 0.1))
after_photometry = consistent_z & mask_phot_err
n_after_photometry = np.count_nonzero(after_photometry)

mask_valid_mags = ((u_dered_qso > -100) & (u_dered_qso < 100) &
                   (g_dered_qso > -100) & (g_dered_qso < 100) &
                   (r_dered_qso > -100) & (r_dered_qso < 100) &
                   (i_dered_qso > -100) & (i_dered_qso < 100) &
                   (z_dered_qso > -100) & (z_dered_qso < 100))
master_mask = after_photometry & mask_valid_mags
n_after_valid = np.count_nonzero(master_mask)

r_i_dered_qso = r_dered_qso[master_mask] - i_dered_qso[master_mask]
g_r_dered_qso = g_dered_qso[master_mask] - r_dered_qso[master_mask]
u_g_dered_qso = u_dered_qso[master_mask] - g_dered_qso[master_mask]
i_z_dered_qso = i_dered_qso[master_mask] - z_dered_qso[master_mask]

## Sky Cells Used for the Stellar Sample

In [5]:
ra = np.asarray(qso_data[master_mask]['RA'])
dec = np.asarray(qso_data[master_mask]['DEC'])

# 10 x 10 deg cells; each cell includes its lower edges and excludes its upper edges
ra_edges = np.arange(0, 361, 10)
dec_edges = np.arange(-90, 91, 10)

ra_cell = np.digitize(ra, ra_edges) - 1
dec_cell = np.digitize(dec, dec_edges) - 1

cells = pd.DataFrame({'ra_cell': ra_cell, 'dec_cell': dec_cell})

cell_counts = (
    cells.groupby(['ra_cell', 'dec_cell'])
    .size()
    .reset_index(name='N_quasars')
    .sort_values('N_quasars', ascending=False)
)

cell_counts['ra_min'] = ra_edges[cell_counts['ra_cell']]
cell_counts['ra_max'] = ra_edges[cell_counts['ra_cell'] + 1]
cell_counts['dec_min'] = dec_edges[cell_counts['dec_cell']]
cell_counts['dec_max'] = dec_edges[cell_counts['dec_cell'] + 1]

selected_cells = cell_counts.head(20).copy()

selected_cells[['ra_min','ra_max','dec_min','dec_max','N_quasars']]

,ra_min,ra_max,dec_min,dec_max,N_quasars
13,20,30,0,10,2645
7,10,20,0,10,2538
2,0,10,0,10,2530
43,130,140,20,30,2377
9,10,20,20,30,2326
60,150,160,30,40,2199
162,350,360,0,10,2176
52,140,150,30,40,2171
36,120,130,30,40,2083
158,340,350,20,30,2062


In [6]:
def cell_area_deg2(ra_min, ra_max, dec_min, dec_max):
    dec_min_rad = np.radians(dec_min)
    dec_max_rad = np.radians(dec_max)
    return (ra_max - ra_min) * (np.sin(dec_max_rad) - np.sin(dec_min_rad)) * (180 / np.pi)

selected_cells['area_deg2'] = selected_cells.apply(
    lambda row: cell_area_deg2(row['ra_min'], row['ra_max'], row['dec_min'], row['dec_max']),
    axis=1
)

total_area = selected_cells['area_deg2'].sum()
print(f'Nominal area of the 20 sky cells: {total_area:.1f} deg²')
print(selected_cells[['ra_min','ra_max','dec_min','dec_max','area_deg2']])

Nominal area of the 20 sky cells: 1740.5 deg²
     ra_min  ra_max  dec_min  dec_max  area_deg2
13       20      30        0       10  99.493077
7        10      20        0       10  99.493077
2         0      10        0       10  99.493077
43      130     140       20       30  90.515790
9        10      20       20       30  90.515790
60      150     160       30       40  81.811274
162     350     360        0       10  99.493077
52      140     150       30       40  81.811274
36      120     130       30       40  81.811274
158     340     350       20       30  90.515790
59      150     160       20       30  90.515790
131     240     250       30       40  81.811274
137     250     260       30       40  81.811274
109     210     220       40       50  70.620964
100     200     210       30       40  81.811274
14       20      30       10       20  96.470030
92      190     200       30       40  81.811274
132     240     250       40       50  70.620964
85      180     190    

The 20 sky cells above were used to construct the spatial WHERE clause
in `SQL/Star_Catalog_Query.sql`, which was submitted to SDSS CasJobs to
download the corresponding stellar catalog (see `Data/README.md` for download
instructions).

## Load the Stellar Catalog (from CasJobs) and Correct for Galactic Extinction

In [7]:
stars_filename = os.path.join(data_path, 'Star_Catalog.fit')
stars_data = Table.read(stars_filename, format='fits', hdu=1)

u_star = stars_data['psfMag_u']
g_star = stars_data['psfMag_g']
r_star = stars_data['psfMag_r']
i_star = stars_data['psfMag_i']
z_star = stars_data['psfMag_z']

u_star_err = stars_data['psfMagErr_u']
g_star_err = stars_data['psfMagErr_g']
r_star_err = stars_data['psfMagErr_r']
i_star_err = stars_data['psfMagErr_i']
z_star_err = stars_data['psfMagErr_z']

u_star_extinction = stars_data['extinction_u']
g_star_extinction = stars_data['extinction_g']
r_star_extinction = stars_data['extinction_r']
i_star_extinction = stars_data['extinction_i']
z_star_extinction = stars_data['extinction_z']

u_dered_star = u_star - u_star_extinction
g_dered_star = g_star - g_star_extinction
r_dered_star = r_star - r_star_extinction
i_dered_star = i_star - i_star_extinction
z_dered_star = z_star - z_star_extinction

r_i_dered_star = r_dered_star - i_dered_star
g_r_dered_star = g_dered_star - r_dered_star
u_g_dered_star = u_dered_star - g_dered_star
i_z_dered_star = i_dered_star - z_dered_star

## Stellar Color-Range Cut

In [8]:
mask_valid_colors_star = ((r_i_dered_star > -2) & (r_i_dered_star < 5) &
                           (g_r_dered_star > -2) & (g_r_dered_star < 5) &
                           (u_g_dered_star > -2) & (u_g_dered_star < 5) &
                           (i_z_dered_star > -2) & (i_z_dered_star < 5))

n_before_color_cut = len(r_i_dered_star)

r_i_dered_star = r_i_dered_star[mask_valid_colors_star]
g_r_dered_star = g_r_dered_star[mask_valid_colors_star]
u_g_dered_star = u_g_dered_star[mask_valid_colors_star]
i_z_dered_star = i_z_dered_star[mask_valid_colors_star]

n_after_color_cut = len(r_i_dered_star)
print('Stellar-sample objects before color cut:', n_before_color_cut)
print('Stellar-sample objects after color cut:', n_after_color_cut)
print('Rejected:', n_before_color_cut - n_after_color_cut)

Stellar-sample objects before color cut: 1855955
Stellar-sample objects after color cut: 1854084
Rejected: 1871


## Export Quasar Selection Cuts

In [9]:
qso_cuts = [
    {'step': 0, 'description': 'total_quasars_in_catalog', 'rejected': None, 'remaining': n_total_qso},
    {'step': 1, 'description': 'valid_quasar_classification_and_redshift', 'rejected': n_total_qso - n_after_classification, 'remaining': n_after_classification},
    {'step': 2, 'description': 'no_pipeline_or_pca_redshift_warning', 'rejected': n_after_classification - n_after_zwarning, 'remaining': n_after_zwarning},
    {'step': 3, 'description': 'exclude_pipeline_only_redshifts_above_z5', 'rejected': n_after_zwarning - n_after_pipe_only_high_z, 'remaining': n_after_pipe_only_high_z},
    {'step': 4, 'description': 'consistent_pipeline_and_quasarnet_redshifts', 'rejected': n_after_pipe_only_high_z - n_after_consistent, 'remaining': n_after_consistent},
    {'step': 5, 'description': 'photometric_uncertainty_below_0.1', 'rejected': n_after_consistent - n_after_photometry, 'remaining': n_after_photometry},
    {'step': 6, 'description': 'valid_extinction_corrected_psf_magnitudes', 'rejected': n_after_photometry - n_after_valid, 'remaining': n_after_valid},
]

qso_cuts_df = pd.DataFrame(qso_cuts)
output_path = os.path.join(os.getcwd(), '..', 'Outputs', 'Quasar_Selection_Cuts.csv')
qso_cuts_df.to_csv(output_path, index=False)
qso_cuts_df

,step,description,rejected,remaining
0,0,total_quasars_in_catalog,NaN,750414
1,1,valid_quasar_classification_and_redshift,1759.0,748655
2,2,no_pipeline_or_pca_redshift_warning,111268.0,637387
3,3,exclude_pipeline_only_redshifts_above_z5,158.0,637229
4,4,consistent_pipeline_and_quasarnet_redshifts,3707.0,633522
5,5,photometric_uncertainty_below_0.1,471418.0,162104
6,6,valid_extinction_corrected_psf_magnitudes,0.0,162104


## Export Stellar Color-Range Cut

In [10]:
star_cuts = [
    {'step': 0, 'description': 'total_sources_in_casjobs_star_catalog', 'rejected': None, 'remaining': n_before_color_cut},
    {'step': 1, 'description': 'extinction_corrected_colors_between_-2_and_5', 'rejected': n_before_color_cut - n_after_color_cut, 'remaining': n_after_color_cut},
]

star_cuts_df = pd.DataFrame(star_cuts)
star_output_path = os.path.join(os.getcwd(), '..', 'Outputs', 'Star_Selection_Cuts_Python.csv')
star_cuts_df.to_csv(star_output_path, index=False)
star_cuts_df

,step,description,rejected,remaining
0,0,total_sources_in_casjobs_star_catalog,NaN,1855955
1,1,extinction_corrected_colors_between_-2_and_5,1871.0,1854084


## Export Extinction-Corrected Colors

In [11]:
np.savez(os.path.join(os.getcwd(), '..', 'Outputs', 'Quasar_Colors.npz'),
         r_i_dered_qso=r_i_dered_qso,
         g_r_dered_qso=g_r_dered_qso,
         u_g_dered_qso=u_g_dered_qso,
         i_z_dered_qso=i_z_dered_qso,
         Z_PCA=qso_data['Z_PCA'][master_mask])

np.savez(os.path.join(os.getcwd(), '..', 'Outputs', 'Star_Colors.npz'),
         r_i_dered_star=r_i_dered_star,
         g_r_dered_star=g_r_dered_star,
         u_g_dered_star=u_g_dered_star,
         i_z_dered_star=i_z_dered_star)